In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsssmartdata2698")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/olist_order_items_dataset.csv"

In [0]:
df_order_items = spark.read.option("header", True)\
                           .option("inferSchema", True)\
                           .csv(ruta)


In [0]:
order_items_schema = StructType(fields=[
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True)
])

In [0]:
df_order_items_final = spark.read\
.option("header", True)\
.schema(order_items_schema)\
.csv(ruta)


In [0]:
order_items_selected_df = df_order_items_final.select(
    col("order_id"),
    col("order_item_id"),
    col("product_id"),
    col("seller_id"),
    col("shipping_limit_date"),
    col("price"),
    col("freight_value")
)

In [0]:
order_items_renamed_df = order_items_selected_df

In [0]:
order_items_final_df = order_items_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
order_items_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.order_items_raw")